<a href="https://colab.research.google.com/github/Sergiosampjr/Sergiosampjr/blob/main/01_construindo_rag_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Aula 7 — Laboratório Prático de RAG
## Notebook 1 — Construindo um RAG do zero

**Objetivo:** construir um pipeline RAG simples e completo no Google Colab, entendendo cada etapa:

`documentos → chunking → embeddings → índice vetorial → recuperação → contexto → LLM → resposta`

Ao final deste notebook, você terá:
- um pequeno corpus de documentos;
- chunks com metadados;
- embeddings multilíngues;
- um índice FAISS;
- uma função de busca semântica;
- uma função `rag()` que recupera contexto e gera uma resposta;
- artefatos salvos para os próximos notebooks.

### Recomendação de execução
No Colab, use **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU** quando disponível.  
O notebook também funciona em CPU, mas a geração de respostas será mais lenta.


## 1. Instalação das dependências

In [ ]:

!pip -q install "sentence-transformers>=3.0" "faiss-cpu>=1.8" "transformers>=4.45" "accelerate>=0.34"


## 2. Imports e configuração

In [ ]:

from pathlib import Path
import json
import re
import random

import faiss
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## 3. Corpus de exemplo

Para que todos os participantes consigam executar o laboratório sem depender de downloads externos de PDFs, usaremos um corpus pequeno já incluído no notebook.

O corpus descreve conceitos básicos de RAG. Nos próximos notebooks, esse corpus será usado para criar perguntas de avaliação e testes automatizados.


In [ ]:

documents = [
    {
        "doc_id": "doc_01",
        "title": "Introdução ao RAG",
        "text": """
Retrieval-Augmented Generation, ou RAG, é uma arquitetura que combina recuperação de informação
com modelos de linguagem. Em vez de responder apenas com base no conhecimento interno do modelo,
o sistema consulta uma base externa, recupera trechos relevantes e os envia ao modelo como contexto.
Essa abordagem pode aumentar a precisão factual e facilitar a atualização do conhecimento sem
necessidade de treinar novamente o modelo de linguagem.
"""
    },
    {
        "doc_id": "doc_02",
        "title": "Embeddings",
        "text": """
Embeddings são representações vetoriais de dados, como palavras, frases ou documentos.
Textos semanticamente parecidos tendem a produzir vetores próximos no espaço vetorial.
Em sistemas RAG, embeddings são usados para representar tanto os documentos quanto a pergunta
do usuário, permitindo recuperar trechos semanticamente relacionados à consulta.
"""
    },
    {
        "doc_id": "doc_03",
        "title": "Busca vetorial e FAISS",
        "text": """
FAISS é uma biblioteca para busca eficiente de similaridade entre vetores.
Em um pipeline RAG, os embeddings dos chunks podem ser armazenados em um índice FAISS.
Quando uma pergunta é recebida, seu embedding é comparado aos vetores armazenados.
O sistema retorna os chunks mais próximos segundo uma medida de similaridade.
"""
    },
    {
        "doc_id": "doc_04",
        "title": "Chunking",
        "text": """
Chunking é o processo de dividir documentos extensos em trechos menores.
Chunks muito pequenos podem perder contexto, enquanto chunks muito grandes podem trazer
informação irrelevante e aumentar o custo de processamento. O tamanho do chunk e a sobreposição
entre trechos são hiperparâmetros importantes em um sistema RAG.
"""
    },
    {
        "doc_id": "doc_05",
        "title": "Ground Truth e avaliação",
        "text": """
Ground Truth é um conjunto de referências usadas para verificar o comportamento esperado do sistema.
Na avaliação de RAG, pode incluir perguntas, respostas de referência e documentos considerados
relevantes. Esse material permite calcular métricas de recuperação e de qualidade das respostas,
além de apoiar testes automatizados de regressão.
"""
    },
]

print("Quantidade de documentos:", len(documents))
for doc in documents:
    print(f"- {doc['doc_id']}: {doc['title']}")



## 4. Chunking

Vamos implementar o chunking manualmente para visualizar claramente o que acontece.

Nesta aula:
- `chunk_size` = quantidade aproximada de palavras por chunk;
- `overlap` = quantidade de palavras reaproveitadas entre chunks consecutivos.

> Em sistemas de produção, o chunking normalmente é feito por tokens e pode considerar estrutura semântica, títulos, seções e metadados.


In [ ]:

def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def chunk_text(text: str, chunk_size: int = 55, overlap: int = 12):
    words = normalize_text(text).split()

    if overlap >= chunk_size:
        raise ValueError("overlap deve ser menor que chunk_size.")

    chunks = []
    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))

        if end == len(words):
            break

        start = end - overlap

    return chunks


CHUNK_SIZE = 55
OVERLAP = 12

chunks = []

for doc in documents:
    doc_chunks = chunk_text(
        doc["text"],
        chunk_size=CHUNK_SIZE,
        overlap=OVERLAP
    )

    for i, chunk in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{i}",
            "doc_id": doc["doc_id"],
            "title": doc["title"],
            "text": chunk
        })

print("Total de chunks:", len(chunks))
chunks[:3]



### Exercício rápido

Antes de continuar, altere temporariamente:
- `CHUNK_SIZE` para `30`;
- `OVERLAP` para `5`.

Observe:
1. quantos chunks passam a ser gerados;
2. se os trechos continuam compreensíveis;
3. como uma configuração menor fragmenta mais os documentos.

Depois, retorne aos valores originais para prosseguir.


## 5. Embeddings

In [ ]:

EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

texts_for_embedding = [
    "passage: " + item["text"]
    for item in chunks
]

embeddings = embedding_model.encode(
    texts_for_embedding,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Formato da matriz:", embeddings.shape)
print("Dimensão de cada embedding:", embeddings.shape[1])



### O que acabou de acontecer?

Cada chunk foi convertido em um vetor numérico.

Como os embeddings foram normalizados, podemos usar **produto interno** no FAISS como aproximação da **similaridade do cosseno**:
- valor mais alto → maior similaridade;
- valor mais baixo → menor similaridade.


## 6. Criando o índice vetorial com FAISS

In [ ]:

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Vetores armazenados no índice:", index.ntotal)


## 7. Função de recuperação

In [ ]:

def retrieve(question: str, k: int = 3):
    query = "query: " + question

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        item = chunks[int(idx)].copy()
        item["score"] = float(score)
        results.append(item)

    return results


In [ ]:

question = "Qual é a função do FAISS em um sistema RAG?"

results = retrieve(question, k=3)

for rank, item in enumerate(results, start=1):
    print(f"\n#{rank} | score={item['score']:.4f}")
    print("Documento:", item["doc_id"], "-", item["title"])
    print("Trecho:", item["text"])



### Ponto de validação

Antes de usar uma LLM, já conseguimos avaliar parte do sistema:

> **A recuperação encontrou o documento correto?**

Essa distinção será importante no Notebook 2, porque um RAG pode falhar em pelo menos dois lugares:
1. o retriever pode buscar contexto inadequado;
2. a LLM pode produzir uma resposta inadequada mesmo recebendo bom contexto.


## 8. Carregando uma LLM pequena

In [ ]:

LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Modelo carregado:", LLM_MODEL)



Usaremos um modelo pequeno para manter o laboratório gratuito e executável no Colab.

O objetivo desta aula **não é maximizar a qualidade da LLM**, mas compreender e validar o pipeline RAG.


## 9. Montando o prompt com contexto recuperado

In [ ]:

def build_prompt(question: str, retrieved_chunks):
    context = "\n\n".join(
        [
            f"[{item['doc_id']} | {item['title']}]\n{item['text']}"
            for item in retrieved_chunks
        ]
    )

    system_message = (
        "Você é um assistente didático. "
        "Responda em português usando somente as informações presentes no contexto. "
        "Se o contexto não contiver a resposta, diga explicitamente que não há informação suficiente."
    )

    user_message = f"""
CONTEXTO:
{context}

PERGUNTA:
{question}

Responda de forma objetiva e cite, entre colchetes, o doc_id usado como fonte.
"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    return messages


## 10. Geração da resposta

In [ ]:

def generate_answer(messages, max_new_tokens: int = 180):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    ).strip()


## 11. Função RAG completa

In [ ]:

def rag(question: str, k: int = 3):
    retrieved = retrieve(question, k=k)
    messages = build_prompt(question, retrieved)
    answer = generate_answer(messages)

    return {
        "question": question,
        "answer": answer,
        "contexts": retrieved
    }


In [ ]:

result = rag("O que são embeddings e para que eles servem em um sistema RAG?")

print("PERGUNTA:")
print(result["question"])

print("\nRESPOSTA:")
print(result["answer"])

print("\nCONTEXTO RECUPERADO:")
for item in result["contexts"]:
    print(
        f"- {item['doc_id']} | {item['title']} | score={item['score']:.4f}"
    )


## 12. Testando perguntas diferentes

In [ ]:

test_questions = [
    "O que é RAG?",
    "Qual é a função do FAISS?",
    "O que significa chunking?",
    "Para que serve o Ground Truth?",
]

for q in test_questions:
    print("=" * 90)
    result = rag(q, k=3)
    print("Pergunta:", q)
    print("Resposta:", result["answer"])
    print("Docs recuperados:", [c["doc_id"] for c in result["contexts"]])
    print()



## 13. Teste fora do domínio

Agora faremos uma pergunta cuja resposta **não existe** no corpus.

Um comportamento desejável é que o modelo reconheça a ausência de evidência, em vez de inventar uma resposta.


In [ ]:

out_of_domain = rag("Qual é a capital da Austrália?", k=3)

print("Resposta:")
print(out_of_domain["answer"])

print("\nContextos recuperados:")
for item in out_of_domain["contexts"]:
    print(item["doc_id"], "-", item["title"], f"({item['score']:.4f})")



### Pergunta para reflexão

Mesmo quando nenhuma informação relevante existe, o retriever ainda retorna os vetores "mais próximos".

Isso significa que:

> **top-k sempre retorna resultados, mas isso não garante que esses resultados sejam realmente relevantes.**

Nos próximos notebooks veremos como detectar esse tipo de problema usando métricas e testes.


## 14. Salvando os artefatos

In [ ]:

with open(ARTIFACT_DIR / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

faiss.write_index(index, str(ARTIFACT_DIR / "faiss.index"))

metadata = {
    "embedding_model": EMBEDDING_MODEL,
    "llm_model": LLM_MODEL,
    "num_documents": len(documents),
    "num_chunks": len(chunks),
    "embedding_dimension": int(embeddings.shape[1]),
}

with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

rag_config = {
    "chunk_size": CHUNK_SIZE,
    "overlap": OVERLAP,
    "top_k": 3,
    "seed": SEED,
}

with open(ARTIFACT_DIR / "rag_config.json", "w", encoding="utf-8") as f:
    json.dump(rag_config, f, ensure_ascii=False, indent=2)

print("Artefatos salvos em:", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    print("-", p.name)



## 15. Checkpoint da aula

Ao final deste notebook, você implementou:

- [x] corpus de documentos;
- [x] chunking;
- [x] embeddings;
- [x] índice FAISS;
- [x] busca semântica;
- [x] recuperação top-k;
- [x] construção de contexto;
- [x] geração com LLM;
- [x] função RAG completa;
- [x] persistência dos artefatos.

### Arquitetura obtida

```text
Pergunta
   ↓
Embedding da pergunta
   ↓
FAISS
   ↓
Top-k chunks
   ↓
Contexto + pergunta
   ↓
LLM
   ↓
Resposta
```



# Atividade prática

Faça **duas alterações**, uma de cada vez:

1. altere `k` de `3` para `1`;
2. altere `CHUNK_SIZE` e recrie os chunks e o índice.

Para cada alteração, teste novamente estas perguntas:

- "O que é RAG?"
- "Qual é a função do FAISS?"
- "O que significa chunking?"

Registre:
- quais documentos foram recuperados;
- se a resposta melhorou ou piorou;
- uma hipótese para explicar o comportamento observado.

> Não é necessário encontrar a "melhor" configuração. O objetivo é perceber que parâmetros do pipeline alteram o comportamento do sistema.

---

## Próximo notebook

**Notebook 2 — Avaliando o RAG**

Vamos transformar a observação manual em uma avaliação sistemática usando:
- Ground Truth;
- métricas de recuperação;
- métricas de qualidade da resposta;
- comparação entre configurações.
